In [1]:
from __future__ import annotations
from typing import List, Dict, Optional, Any, Tuple
import time
import torch
from flair.data import Sentence
from flair.nn import Classifier
from flair.models import EntityMentionLinker
import json 
import requests # <-- NEW IMPORT

EUTILS_BASE = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"

def _get(url: str, params: Dict, tries: int = 3, backoff: float = 0.7) -> requests.Response:
    """Simple retrying GET with exponential backoff."""
    for i in range(tries):
        r = requests.get(url, params=params, timeout=30)
        if r.ok:
            return r
        if i < tries - 1:
            time.sleep(backoff * (2 ** i))
    r.raise_for_status()
    return r  # pragma: no cover

def mesh_descriptor_to_cids(
    mesh_did: str,
    api_key: Optional[str] = None,
    email: Optional[str] = None,
    tool: str = "mesh2cid_example"
) -> List[int]:
    """
    Convert a MeSH Descriptor ID (e.g., 'D000082') to PubChem CIDs via:
      1) ESearch MeSH -> numeric UID
      2) ELink MeSH (UID) -> PubChem Compound (pccompound)
    Returns a sorted, de-duplicated list of CIDs (ints). May be empty if no link.
    """

    # 1) MeSH Dxxxxxx -> numeric MeSH UID
    esearch_params = {
        "db": "mesh",
        "retmode": "json",
        "term": f"{mesh_did}[MeSH Unique ID]",
        "tool": tool
    }
    if api_key: esearch_params["api_key"] = api_key
    if email:   esearch_params["email"] = email

    r = _get(f"{EUTILS_BASE}/esearch.fcgi", esearch_params)
    data = r.json()
    idlist = data.get("esearchresult", {}).get("idlist", [])
    if not idlist:
        return []

    mesh_uid = idlist[0]  # use the first/only UID

    # 2) Link that MeSH UID to PubChem Compound (pccompound) -> CIDs
    elink_params = {
        "dbfrom": "mesh",
        "db": "pccompound",
        "retmode": "json",
        "id": mesh_uid,
        "tool": tool
    }
    if api_key: elink_params["api_key"] = api_key
    if email:   elink_params["email"] = email

    r2 = _get(f"{EUTILS_BASE}/elink.fcgi", elink_params)
    d2 = r2.json()

    cids = []
    for ls in d2.get("linksets", []):
        for ldb in (ls.get("linksetdbs") or []):
            # Links are CIDs when db=pccompound
            cids.extend(ldb.get("links", []) or [])

    # de-duplicate + sort as ints
    try:
        cids_int = sorted({int(x) for x in cids})
    except Exception:
        cids_int = sorted({int(str(x)) for x in cids if str(x).isdigit()})

    return cids_int

# Module-level cache so models are loaded once and pinned to GPUs
_MODELS: Dict[Tuple[str, str, int, int, bool], Dict[str, Any]] = {}

EntityTuple = Tuple[str, int, int, Optional[float], Optional[float]]
EntityValue = Dict[str, Any]  # e.g., {"cids": [2244], "mentions": [('Aspirin', ...)]}
MergedResultDict = Dict[Optional[str], EntityValue]


def _device_for(idx: int) -> torch.device:
    if torch.cuda.is_available() and torch.cuda.device_count() > idx:
        return torch.device(f"cuda:{idx}")
    return torch.device("cpu")

def _load_and_prepare_models(
    ner_model_name: str = "hunflair2",
    linker_model_name: str = "chemical-linker",
    ner_gpu: int = 0,
    linker_gpu: int = 1,
    use_fp16: bool = False,
    warmup: bool = True,
    warmup_text: str = "Test chemical aspirin."
):
    """
    Load models once, move them to devices, optionally convert to fp16, and warm up.
    Returns tuple (ner, linker, ner_device, linker_device).
    """
    key = (ner_model_name, linker_model_name, ner_gpu, linker_gpu, use_fp16)
    if key in _MODELS:
        return _MODELS[key]["ner"], _MODELS[key]["linker"], _MODELS[key]["ner_device"], _MODELS[key]["linker_device"]

    ner_device = _device_for(ner_gpu)
    linker_device = _device_for(linker_gpu)

    # Optional: make cuDNN faster for fixed-size inputs
    try:
        torch.backends.cudnn.benchmark = True
    except Exception:
        pass

    # Load models (first-time download may take time)
    ner = Classifier.load(ner_model_name)
    linker = EntityMentionLinker.load(linker_model_name)

    # Move to requested devices
    ner.to(ner_device)
    linker.to(linker_device)

    # Optionally convert model params to half precision (fp16) for inference speed/memory.
    # Not every Flair model or custom layer will be fp16-safe; test before enabling in production.
    if use_fp16 and ner_device.type == "cuda":
        try:
            ner.half()
        except Exception:
            # If .half() fails, we keep full precision
            pass
    if use_fp16 and linker_device.type == "cuda":
        try:
            linker.half()
        except Exception:
            pass

    ner.eval()
    linker.eval()

    # Warm up: a single tiny prediction to initialize CUDA kernels and JIT paths.
    if warmup:
        try:
            s = Sentence(warmup_text)
            with torch.inference_mode():
                # small warmup on ner (runs on ner_device)
                ner.predict(s)
                # then warmup linker (runs on linker_device)
                linker.predict(s)
        except Exception:
            # ignore warmup failures
            pass

    _MODELS[key] = {
        "ner": ner,
        "linker": linker,
        "ner_device": ner_device,
        "linker_device": linker_device,
    }
    # Debug print to help you verify devices - keep minimal in library usage
    try:
        nd = next(ner.parameters()).device if any(True for _ in ner.parameters()) else None
    except Exception:
        nd = None
    try:
        ld = next(linker.parameters()).device if any(True for _ in linker.parameters()) else None
    except Exception:
        ld = None
    print(f"[model] ner -> {ner_device} (params on {nd}), linker -> {linker_device} (params on {ld})")
    return ner, linker, ner_device, linker_device

def extract_chemicals_with_nen_batch(
    texts: List[str],
    ner_gpu: int = 0,
    linker_gpu: int = 1,
    ner_model_name: str = "hunflair2",
    linker_model_name: str = "chemical-linker",
    use_fp16: bool = False,
    verbose: bool = False,
    ner_mini_batch_size: int = 32, 
) -> List[MergedResultDict]: # <-- MODIFIED Return Type
    """
    Batch processing version of chemical extraction + NEN.
    
    Processes a list of texts and returns a list of results, where each
    item in the list is a single dictionary containing all entities
    for the corresponding text.
    
    Args:
        (Same as before)
        
    Returns:
        List[MergedResultDict]: A list of dictionaries. Each dictionary
                                maps KB ID (or None) to a dict containing
                                'cids' and 'mentions'.
    """
    if not isinstance(texts, list):
        raise TypeError("texts must be a list of strings")
    if not texts:
        return []

    # load models once per combination of args
    ner, linker, ner_device, linker_device = _load_and_prepare_models(
        ner_model_name=ner_model_name,
        linker_model_name=linker_model_name,
        ner_gpu=ner_gpu,
        linker_gpu=linker_gpu,
        use_fp16=use_fp16,
        warmup=True,
    )

    # Create a Sentence object for each text
    sentences = [Sentence(text) for text in texts if text.strip()]
    if not sentences:
        # Return empty dict for each input text
        return [{} for _ in texts] 

    t0 = time.time()
    # Run NER prediction on the whole batch
    with torch.inference_mode():
        ner.predict(sentences, mini_batch_size=ner_mini_batch_size)
    t1 = time.time()

    # Run Linker prediction on the whole batch
    with torch.inference_mode():
        linker.predict(sentences)
    t2 = time.time()

    if verbose and sentences:
        total_sents = len(sentences)
        avg_ner = (t1-t0) / total_sents if total_sents else 0
        avg_linker = (t2-t1) / total_sents if total_sents else 0
        print(f"[timing-batch] total_sents: {total_sents}, "
              f"ner: {t1-t0:.3f}s ({avg_ner*1000:.2f} ms/sent), "
              f"linker: {t2-t1:.3f}s ({avg_linker*1000:.2f} ms/sent), "
              f"total: {t2-t0:.3f}s")
    
    # Create a dictionary of results from non-empty sentences
    results_map: Dict[str, MergedResultDict] = {}
    
    for sent in sentences:
        # Each sentence gets one merged dictionary
        chemicals_for_this_sent: MergedResultDict = {}
        
        for span in sent.get_spans("ner"):
            if span.tag is None:
                continue
            if span.tag.lower() != "chemical":
                continue

            # character offsets (best-effort)
            try:
                start_char = int(getattr(span.tokens[0], "start_position", None) or -1)
            except Exception:
                start_char = -1
            try:
                end_char = int(getattr(span.tokens[-1], "end_position", None) or -1)
            except Exception:
                end_char = -1

            ner_score = float(span.score) if getattr(span, "score", None) is not None else None

            kb_id: Optional[str] = None
            linker_score: Optional[float] = None
            
            link_labels = span.get_labels("link")
            if link_labels:
                top = max(link_labels, key=lambda l: l.score)
                kb_id = top.value
                linker_score = float(top.score)
            
            # Create the tuple of entity info
            entity_tuple: EntityTuple = (span.text, start_char, end_char, ner_score, linker_score)

            # Append this tuple to the list for this kb_id
            if kb_id not in chemicals_for_this_sent:
                chemicals_for_this_sent[kb_id] = {"cids": [], "mentions": []}
            chemicals_for_this_sent[kb_id]["mentions"].append(entity_tuple)
        
        # Store the complete, merged dictionary for this sentence
        results_map[sent.to_original_text()] = chemicals_for_this_sent

    # Build the final list in the original order
    all_results: List[MergedResultDict] = []
    for text in texts:
        # Get results from the map, or an empty dict for empty/whitespace-only strings
        all_results.append(results_map.get(text, {})) 

    return all_results


def merge_batch_results(batch_results: List[MergedResultDict]) -> MergedResultDict:
    """
    Merges a list of result dictionaries into a single dictionary.
    
    It combines all entities found across all texts, grouping them
    by their KB ID.
    
    Args:
        batch_results: The output list from extract_chemicals_with_nen_batch().
        
    Returns:
        A single MergedResultDict containing all entities from all texts.
    """
    final_merged_dict: MergedResultDict = {}
    
    # Iterate over each dictionary in the list
    for result_dict in batch_results:
        # Iterate over each (kb_id, value_dict) pair in that dictionary
        for kb_id, value_dict in result_dict.items():
            
            # If the kb_id is not in our final dict, add it with an empty structure
            if kb_id not in final_merged_dict:
                final_merged_dict[kb_id] = {"cids": [], "mentions": []}
                
            # Extend the list of mentions for this kb_id
            final_merged_dict[kb_id]["mentions"].extend(value_dict["mentions"])
            
    return final_merged_dict

def ner_text(texts_list: List[str]) -> dict:

    # Note: This call should use the cached models from the single run
    results = extract_chemicals_with_nen_batch(
        texts_list, 
        ner_gpu=0, 
        linker_gpu=1, 
        verbose=True, 
        ner_mini_batch_size=4 
    )

    # Call the new merge function
    final_merged_data = merge_batch_results(results)

    NCBI_EMAIL = "laozc@student.ubc.ca"
    NCBI_TOOL = "flair_nen_to_cid"
    
    # Create a cache so we don't re-fetch for the same ID
    mesh_id_cache: Dict[str, List[int]] = {}

    # Iterate over a copy of keys, as we are modifying the dict
    for kb_id in list(final_merged_data.keys()):
        if kb_id is None or not kb_id.startswith("MESH:"):
            continue
        
        # Strip prefix: "MESH:D001241" -> "D001241"
        mesh_did = kb_id[5:] 
        
        if not mesh_did:
            continue
        
        if mesh_did not in mesh_id_cache:
            print(f"Querying E-Utils for: {mesh_did} ({kb_id})")
            try:
                cids = mesh_descriptor_to_cids(
                    mesh_did=mesh_did, 
                    email=NCBI_EMAIL, 
                    tool=NCBI_TOOL
                )
                mesh_id_cache[mesh_did] = cids
                # Be polite to NCBI - ~3 requests/sec
                time.sleep(0.34) 
            except Exception as e:
                print(f"Error fetching CIDs for {mesh_did}: {e}")
                mesh_id_cache[mesh_did] = []
        
        # Populate the CIDs from the cache
        final_merged_data[kb_id]["cids"] = mesh_id_cache[mesh_did]

    return final_merged_data


texts_list = [
    "Biotin was detected in human hair at 2.5 ng/mg using LC-MS/MS.",
    "This study analyzes the effects of paracetamol and ibuprofen.",
    "Imidazolyl carboxylic acid derivative here.", # Should link to None
    "", # Test empty string
    "Aspirin and aspirin are the same.", # Test multiple mentions of same ID
    "We found Acetylsalicylate but not salicylate." # Test multiple, different IDs
]

ner_text(texts_list)

/home/chen/GNPS_SIM/agent/db_form_agent_system/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2025-11-04 21:43:33,384 SequenceTagger predicts: Dictionary with 21 tags: O, S-Chemical, B-Chemical, E-Chemical, I-Chemical, S-Gene, B-Gene, E-Gene, I-Gene, S-Disease, B-Disease, E-Disease, I-Disease, S-Species, B-Species, E-Species, I-Species, S-CellLine, B-CellLine, E-CellLine, I-CellLine
[model] ner -> cuda:0 (params on cuda:0), linker -> cuda:1 (params on None)
[timing-batch] total_sents: 5, ner: 0.020s (3.99 ms/sent), linker: 3.382s (676.34 ms/sent), total: 3.402s
Querying E-Utils for: D001710 (MESH:D001710)
Querying E-Utils for: D000082 (MESH:D000082)
Querying E-Utils for: D007052 (MESH:D007052)
Querying E-Utils for: C004806 (MESH:C004806)
Querying E-Utils for: D001241 (MESH:D001241)
Querying E-Utils for: D012459 (MESH:D012459)


{'MESH:D001710': {'cids': [253,
   171548,
   638035,
   5315463,
   6347547,
   6419877,
   6560210,
   6604192,
   6604653,
   7048618,
   7048619,
   7048620,
   7048621,
   10444559,
   11184136,
   24892859],
  'mentions': [('Biotin', -1, 6, 0.999997615814209, 216.00546264648438)]},
 'MESH:D000082': {'cids': [1983,
   10441890,
   10464506,
   10654423,
   10749384,
   10986410,
   11321051,
   11506497],
  'mentions': [('paracetamol',
    35,
    46,
    0.9999978542327881,
    221.3515167236328)]},
 'MESH:D007052': {'cids': [3672, 39912, 5338317, 6919043, 6926382, 23662458],
  'mentions': [('ibuprofen', 51, 60, 0.9999978542327881, 215.7772979736328)]},
 'MESH:C004806': {'cids': [162033],
  'mentions': [('Imidazolyl carboxylic acid',
    -1,
    26,
    0.9999974171320597,
    194.1741180419922)]},
 'MESH:D001241': {'cids': [2244, 6247, 450661, 3434975],
  'mentions': [('Aspirin', -1, 7, 0.9999979734420776, 213.3346405029297),
   ('aspirin', 12, 19, 0.9999978542327881, 213.334640

In [2]:

# -------------------- END OF TOOL DEFINITION --------------------
# import /home/chen/GNPS_SIM/agent/db_form_agent_system/code/chemical_ner_tool.py

import json
import sys

sys.path.append("../../code")

from chemical_ner_tool import ChemicalNerTool  # noqa: E402


if __name__ == "__main__":
    
    tool = ChemicalNerTool(ncbi_email="you@ubc.ca")

    res = tool.run({
        "texts": ["Biotin was detected in serum at 2.5 ng/mL.", "We spiked pyridoxine into plasma."],
        "get_synonyms": False  # set False to skip PubChem synonym lookups
    })
    print(res)

{
  "error": false,
  "merged": {
    "MESH:D001710": {
      "synonyms": [],
      "mentions": [
        {
          "text": [
            "Biotin"
          ]
        }
      ]
    },
    "MESH:D011736": {
      "synonyms": [],
      "mentions": [
        {
          "text": [
            "pyridoxine"
          ]
        }
      ]
    }
  },
  "mention_count": 2
}
